<h1>Ideias e BrainStorm de Desenvolvimento</h1>

* Oque devemos olhar
* Quais validações são importantes para o cliente

In [1]:
import polars as pl

In [4]:
bases = '/Users/PedroFranca/Desktop/Projeto dashboard Apartamento/pasta-git-projeto-data-viz/data_viz_projeto_pbl_final.grupo-davi_pena-pedro_franca-ulisses_pena/bases/'

cartoes = pl.read_csv(f'{bases}cartoes.csv')
commits = pl.read_csv(f'{bases}commits.csv')
grupos = pl.read_csv(f'{bases}grupos.csv')
kanban_eventos = pl.read_csv(f'{bases}kanban_eventos.csv')
merge_requests = pl.read_csv(f'{bases}merge_requests.csv')
pessoas = pl.read_csv(f'{bases}pessoas.csv')
quadro_colunas = pl.read_csv(f'{bases}quadro_colunas.csv')
sprints = pl.read_csv(f'{bases}sprints.csv')

In [24]:
kanban_eventos

grupo,cartao_numero,acao,coluna,pessoa_id,ocorrido_em
str,i64,str,str,str,str
"""G01""",1,"""add""","""DOCUMENTATION""","""G01-A13""","""2026-04-22T17:39:47.412Z"""
"""G01""",1,"""add""","""SIZE G""","""G01-A13""","""2026-04-22T17:39:47.412Z"""
"""G01""",1,"""add""","""Backlog""","""G01-A13""","""2026-04-22T17:40:46.797Z"""
"""G01""",1,"""add""","""NEGÓCIOS""","""G01-A13""","""2026-04-22T17:49:21.492Z"""
"""G01""",1,"""remove""","""Backlog""","""G01-A13""","""2026-04-22T17:49:52.195Z"""
…,…,…,…,…,…
"""G03""",455,"""add""","""Waiting Review""","""G03-A12""","""2026-06-26T20:27:44.116Z"""
"""G03""",455,"""remove""","""Review""","""G03-A12""","""2026-06-26T20:27:44.116Z"""
"""G03""",455,"""add""","""Review""","""G03-A18""","""2026-06-26T20:43:50.260Z"""


In [26]:
kanban = kanban_eventos.with_columns(
    pl.concat_str(pl.col('cartao_numero').cast(pl.String),pl.col('grupo')).alias('chave')
).join(
    cartoes.with_columns(
        pl.concat_str(pl.col('cartao_numero').cast(pl.String),pl.col('grupo')).alias('chave')
    ),
    on=pl.col('chave'),
    how='left'
)

kanban.write_parquet('/Users/PedroFranca/Desktop/Projeto dashboard Apartamento/pasta-git-projeto-data-viz/data_viz_projeto_pbl_final.grupo-davi_pena-pedro_franca-ulisses_pena/parquets/kanban.parquet')

In [29]:
kanban.group_by(
    pl.col('grupo'),
    pl.col('sprint')
).agg(
    pl.len().alias('quantidade cards')
).with_columns(
    pl.coalesce(pl.col('sprint'),pl.lit('Sprint Não Informado')).alias('sprint')
)

grupo,sprint,quantidade cards
str,str,u32
"""G02""","""Sprint 02""",946
"""G01""","""Sprint Não Informado""",182
"""G02""","""Sprint 05""",1343
"""G03""","""Sprint 01""",579
"""G01""","""Sprint 03""",692
…,…,…
"""G01""","""Sprint 04""",365
"""G03""","""Sprint 04""",1108
"""G01""","""Sprint 01""",452


In [18]:
merge_requests.filter(
    pl.col('autor_id') == '[externo]'
).with_columns(
    pl.when(
        pl.col('autor_id') == '[externo]'
    ).then(
        pl.coalesce(pl.col('merged_por_id'),pl.col('revisores_ids'))
    ).alias('tratamento_responsavel_id')
)

grupo,mr_numero,titulo,descricao,situacao,criado_em,atualizado_em,merged_em,fechado_em,branch_origem,branch_destino,autor_id,merged_por_id,revisores_ids,responsaveis_ids,e_rascunho,comentarios,sprint,rotulos,tratamento_responsavel_id
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str
"""G01""",1,"""docs(wad) adiciona introducao …","""Paragrafo introdutorio do WAD""","""merged""","""2026-04-27T17:07:44.357Z""","""2026-04-27T17:26:44.796Z""","""2026-04-27T17:26:43.640Z""",null,"""feat/wad/adicionar-introducao""","""main""","""[externo]""","""G01-A15""",null,null,0,0,null,null,"""G01-A15"""
"""G01""",13,"""docs/wad/redigir-user-storys/#…",null,"""merged""","""2026-04-29T18:20:44.665Z""","""2026-04-30T01:15:06.086Z""","""2026-04-30T01:15:05.396Z""",null,"""docs/wad/redigir-user-storys/#…","""main""","""[externo]""","""G01-A14""",null,null,0,0,null,null,"""G01-A14"""
"""G01""",25,"""Docs/wad/adiciona modelo logic…",null,"""merged""","""2026-05-05T18:32:08.756Z""","""2026-05-06T02:11:25.569Z""","""2026-05-06T02:11:24.961Z""",null,"""docs/wad/adiciona-modelo-logic…","""main""","""[externo]""","""G01-A14""",null,null,0,0,null,null,"""G01-A14"""
"""G01""",29,"""Cria template do arquivo raiz …",null,"""merged""","""2026-05-11T18:19:01.378Z""","""2026-05-11T18:21:15.329Z""","""2026-05-11T18:21:14.702Z""",null,"""feat/scr/migration.sql/adicion…","""main""","""[externo]""","""G01-A14""",null,null,0,0,null,null,"""G01-A14"""
"""G01""",31,"""Feat/g01/reformula template""","""Olhar com atencao""","""merged""","""2026-05-12T17:31:44.874Z""","""2026-05-12T17:35:38.500Z""","""2026-05-12T17:35:37.659Z""",null,"""feat/g01/reformula-template""","""main""","""[externo]""","""G01-A17""",null,null,0,0,null,null,"""G01-A17"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""G02""",46,"""Docs/corrige erros art2""","""Corrige os erros do artefato 2…","""merged""","""2026-05-12T12:16:51.771Z""","""2026-05-12T12:27:00.806Z""","""2026-05-12T12:27:00.146Z""",null,"""docs/corrige-erros-art2""","""main""","""[externo]""","""G02-A12""","""G02-A12""","""[externo]""",0,0,"""Sprint 02""",null,"""G02-A12"""
"""G02""",51,"""Docs/review art5""","""Review do artefato 5 até o mom…","""merged""","""2026-05-13T18:22:34.702Z""","""2026-05-13T18:38:35.094Z""","""2026-05-13T18:38:34.695Z""",null,"""docs/review-art5""","""main""","""[externo]""","""G02-A16""","""G02-A16""","""[externo]""",0,0,"""Sprint 02""",null,"""G02-A16"""
"""G02""",52,"""Docs/review art5 quinta""","""Review dos diagramas, textos e…","""merged""","""2026-05-15T02:42:40.266Z""","""2026-05-15T02:55:54.196Z""","""2026-05-15T02:55:53.652Z""",null,"""docs/review-art5-quinta""","""main""","""[externo]""","""G02-A15""","""G02-A15""","""[externo]""",0,0,"""Sprint 02""",null,"""G02-A15"""
